# ClickHouse из Jupyter

Нужен профиль `clickhouse` (`make up-clickhouse` или `make up-full`).
ClickHouse тут показателен тем, что ходит и в Greenplum, и в MinIO напрямую.

In [ ]:
!pip install -q clickhouse-connect pandas

In [ ]:
import clickhouse_connect
ch = clickhouse_connect.get_client(host="clickhouse", port=8123, username="default", password="clickhouse")
print("ClickHouse", ch.server_version)

## Читаем Greenplum напрямую — табличная функция postgresql()

In [ ]:
ch.query_df("SELECT region, count() c, round(sum(amount),2) total "
            "FROM postgresql('greenplum:5432','gpadmin','sales','gpadmin','gpadmin') "
            "GROUP BY region ORDER BY region")

## Своя таблица MergeTree + загрузка из Greenplum

In [ ]:
ch.command("CREATE TABLE IF NOT EXISTS sales_ch (id Int64, region String, product String, amount Decimal(12,2)) "
           "ENGINE = MergeTree ORDER BY id")
ch.command("INSERT INTO sales_ch SELECT id, region, product, amount "
           "FROM postgresql('greenplum:5432','gpadmin','sales','gpadmin','gpadmin')")
ch.query_df("SELECT count() rows, uniq(region) regions FROM sales_ch")

## Читаем Parquet/Iceberg прямо из MinIO

Коллекция `minio` (endpoint + ключи) задана в `conf/clickhouse/config.d/s3.xml`.
Сначала `make spark-demo` — он положит в MinIO таблицу `analytics.sales_by_region_product`.

Два способа:
- `s3()` — читает parquet-файлы по маске. Просто, но в обход снапшотов Iceberg:
  после перезаписей/удалений может посчитать лишнее.
- `icebergS3()` — разбирает метаданные Iceberg и читает корректное состояние таблицы.

In [ ]:
# По маске — сырые parquet-файлы таблицы:
print(ch.query_df("SELECT count() files_rows, round(sum(total_amount),2) total "
                  "FROM s3(minio, filename='analytics.db/sales_by_region_product/data/*.parquet', format='Parquet')"))

# Нативно как Iceberg:
print(ch.query_df("SELECT region, product, round(total_amount,2) total "
                  "FROM icebergS3(minio, filename='analytics.db/sales_by_region_product') ORDER BY region"))

**Нюанс:** если из Impala делали `DELETE` (ноутбук 03), у таблицы появляются
position-delete файлы, и `icebergS3()` откажется её читать (`positional and equality
deletes are not supported` в ClickHouse 25.x). Лечится перезаписью таблицы — `make spark-demo`.